# PinchBench - Per-Task Deep Dive

Interaktives Notebook fuer Detail-Analyse einzelner PinchBench-Tasks.

**Workflow:**
1. Tasks browsen / filtern.
2. Ein Task auswaehlen.
3. Agent bauen (frisch pro Run), Workspace provisionieren.
4. Mission laufen lassen - jedes Event wird capture'd.
5. Berichte ansehen: tool stats, context evolution, ALLE tool results
   (inkl. Output), final answer, workspace-diff.
6. Grading: automated check + optional LLM-judge - mit per-Kriterium
   breakdown.
7. Failure-Analyse: was fehlt, judge-reasoning, agent-antwort.

Stuetzt sich auf ``notebooks/analysis_lib.py``. Mission-Setup und Grading
spiegeln ``evals/pinchbench/solver.py`` und ``evals/pinchbench/scorer.py``,
aber alles laeuft hier sichtbar im Notebook (nicht im Inspect-AI-Wrapper).


## 1. Setup


In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks'), str(PYTF_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

# Azure env (mirrors evals/run_eval.py setup)
from evals.bridge.azure_config import setup_azure_env, DEFAULT_MODEL
setup_azure_env()

# Reload analysis_lib on every setup cell - picks up edits without kernel restart.
import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib

print('analysis_lib OK')
print(f'  pytaskforce  : {PYTF_ROOT}')
print(f'  default model: {DEFAULT_MODEL}')


## 2. Factory, Executor, Pinchbench-Profil registrieren

Bootstrap-Aufruf laedt die ``configs/`` jedes installierten Agent-Pakets
in den Profile-Loader. PinchBench-Agent muss installiert sein
(``uv pip install -e agents/pinchbench-agent``).


In [ ]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor
from taskforce.application.bootstrap_config_dirs import bootstrap_config_dirs

# Registers agents/pinchbench-agent/configs/ with the ProfileLoader.
registered = bootstrap_config_dirs(force=True)
print(f'Registered config dirs ({len(registered)}):')
for d in registered:
    print(f'  - {d}')

factory  = AgentFactory()
executor = AgentExecutor(factory=factory)
alib.disable_post_mission_learning(executor)

# Will be rebuilt fresh per run. This is a smoke build to verify the
# profile loads + report effective settings (max_steps etc).
async def build_agent():
    a = await factory.create_agent(profile='pinchbench')
    return a

_smoke = await build_agent()
print()
print(f'Pinchbench agent smoke-built OK')
print(f'  Tools                 : {sorted(_smoke.tools.keys())}')
print(f'  Planning strategy     : {_smoke.planning_strategy.name}')
print(f'  Max steps             : {_smoke.max_steps}')
print(f'  Max parallel tools    : {_smoke.max_parallel_tools}')
print(f'  summary_threshold     : {_smoke.message_history_manager._summary_threshold}')
print(f'  tool_result_store thr : {_smoke.tool_executor._result_store_threshold}')
await _smoke.close()


## 3. PinchBench-Tasks laden

Klont die Skill-Repo bei Bedarf, parst alle ``task_*.md`` aus
``evals/pinchbench/skill/tasks/``.


In [ ]:
from evals.pinchbench.loader import load_tasks
import pandas as pd

tasks = load_tasks(suite='all')

df = pd.DataFrame([{
    'id': t.id,
    'category': t.category,
    'grading_type': t.grading_type,
    'timeout_s': t.timeout_seconds,
    'has_grade_fn': bool(t.grade_function),
    'has_rubric': bool(t.llm_judge_rubric),
    'workspace_files_n': len(t.workspace_files),
    'multi_session': bool(t.multi_session_prompts),
    'prompt_len': len(t.prompt),
} for t in tasks])

print(f'Total tasks: {len(tasks)}')
print()
print('Per-category overview:')
print(df.groupby('category').agg(
    n=('id', 'count'),
    automated=('grading_type', lambda s: (s == 'automated').sum()),
    hybrid=('grading_type', lambda s: (s == 'hybrid').sum()),
    llm_judge=('grading_type', lambda s: (s == 'llm_judge').sum()),
).sort_values('n', ascending=False).to_string())


In [ ]:
# Filter tasks - tweak as needed.
FILTER_CATEGORY = None         # e.g. 'meeting_analysis' or None
FILTER_GRADING  = None         # e.g. 'automated' or None
FILTER_SUBSTR   = None         # e.g. 'csv_stock' or None

filtered = tasks
if FILTER_CATEGORY:
    filtered = [t for t in filtered if t.category == FILTER_CATEGORY]
if FILTER_GRADING:
    filtered = [t for t in filtered if t.grading_type == FILTER_GRADING]
if FILTER_SUBSTR:
    filtered = [t for t in filtered if FILTER_SUBSTR in t.id]

print(f'Matching tasks: {len(filtered)}')
print()
for t in filtered[:80]:
    print(f'  {t.id:42s}  [{t.category:18s}]  {t.grading_type:10s}  ws={len(t.workspace_files):2d}  prompt={len(t.prompt):4d}c')
if len(filtered) > 80:
    print(f'  ... +{len(filtered)-80} more (raise the slice if you want)')


## 4. Task auswaehlen + Detail-View

Setz die ID auf den Task den du analysieren willst. Default ist
``task_csv_iris_summary`` (eines der CSV-Hybrid-Beispiele).


In [ ]:
TARGET_TASK_ID = 'task_csv_iris_summary'   # change me

target = next((t for t in tasks if t.id == TARGET_TASK_ID), None)
if target is None:
    raise SystemExit(f'Task {TARGET_TASK_ID!r} not found. Browse above for valid IDs.')

print(f'== {target.id} ==')
print(f'Name        : {target.name}')
print(f'Category    : {target.category}')
print(f'Grading     : {target.grading_type}')
print(f'Timeout     : {target.timeout_seconds}s')
print(f'Workspace   : {target.workspace_files}')
print(f'Multi-session: {bool(target.multi_session_prompts)}')
print()
print('--- PROMPT ---')
print(target.prompt)


In [ ]:
if target.grade_function:
    print('--- AUTOMATED GRADE FUNCTION ---')
    print(target.grade_function[:3000])
    if len(target.grade_function) > 3000:
        print(f'... [+{len(target.grade_function)-3000} chars truncated]')
else:
    print('(no automated grade function for this task)')

print()
if target.llm_judge_rubric:
    print('--- LLM JUDGE RUBRIC ---')
    print(target.llm_judge_rubric[:2500])
    if len(target.llm_judge_rubric) > 2500:
        print(f'... [+{len(target.llm_judge_rubric)-2500} chars truncated]')
else:
    print('(no LLM-judge rubric)')

if target.expected_behavior:
    print()
    print('--- EXPECTED BEHAVIOR ---')
    print(target.expected_behavior[:1500])

if target.grading_criteria:
    print()
    print('--- GRADING CRITERIA ---')
    print(target.grading_criteria[:1500])


## 5. Workspace provisionieren + Prompt augmentieren

Spiegelt ``pinchbench_solver`` 1:1 - Workspace ist ein Temp-Dir mit
allen Fixtures aus ``workspace_files``, Prompt erhaelt den absoluten
Workspace-Pfad als Hinweis.


In [ ]:
from evals.pinchbench.solver import _provision_workspace, _augment_prompt

workspace = _provision_workspace(list(target.workspace_files))
augmented_prompt = _augment_prompt(target.prompt, workspace)

print(f'Workspace : {workspace}')
print()
print('Workspace contents:')
for entry in sorted(workspace.rglob('*')):
    if entry.is_file():
        size_kb = entry.stat().st_size / 1024
        rel = entry.relative_to(workspace)
        print(f'  {rel}  ({size_kb:.1f} KB)')

print()
print(f'Augmented prompt ({len(augmented_prompt)} chars, last 250):')
print(augmented_prompt[-250:])


## 6. Mission ausfuehren

Frischer Agent pro Run (saubere Konversation, kein State-Leak).
``alib.run`` capture'd jedes Event, snapshotted das Workspace
vorher/nachher, und gibt einen ``RunRecord`` zurueck.


In [ ]:
# Fresh build per run.
agent = await build_agent()
# Snapshot any file the agent might write inside the workspace.
SNAP_DIRS = ('',)  # entire workspace root

rec = await alib.run(
    executor, agent, augmented_prompt,
    project_root=workspace,
    snapshot_subdirs=SNAP_DIRS,
    initial_system_prompt_chars=sum(
        len(str(m.get('content',''))) for m in agent.context.messages if m.get('role') == 'system'
    ),
    max_print_events=60,
)
print()
print('=' * 60)
alib.print_summary(rec)


## 7. Berichte

Alle Reports stammen aus ``alib`` - sie zeigen pro Run:

- ``print_tool_stats`` : Wie oft welches Tool, success vs failure.
- ``print_tool_results`` : Jeder Tool-Call mit Argumenten und Output.
- ``print_context_evolution`` : Wie Messages wachsen + Compression-Trigger.
- ``print_sanity_check`` : Ist die Konversation strukturell konsistent?
- ``print_context_view`` : Komplette finale Message-Liste.


In [ ]:
alib.print_tool_stats(rec)


In [ ]:
# All tool results, full output. head=None = show every call.
# Use head=20 to truncate for very long runs.
alib.print_tool_results(rec, head=None)


In [ ]:
alib.print_context_evolution(rec)


In [ ]:
alib.print_sanity_check(rec)


In [ ]:
alib.print_context_view(agent, rec)


## 8. Workspace-Inspektion

Was hat der Agent tatsaechlich auf Disk geschrieben? Vergleich
vorher/nachher, plus Anzeige der neu geschriebenen Datei-Inhalte.


In [ ]:
diff = alib.file_diff_report(rec)
print(f'New files     : {len(diff["new"])}')
for f in diff['new']:
    full = workspace / f
    size = full.stat().st_size if full.exists() else 0
    print(f'  + {f}  ({size} bytes)')
print(f'Modified files: {len(diff["modified"])}')
for f in diff['modified']:
    full = workspace / f
    size = full.stat().st_size if full.exists() else 0
    print(f'  ~ {f}  ({size} bytes)')
print(f'Deleted       : {len(diff["deleted"])}')
print(f'Unchanged     : {len(diff["unchanged"])} (fixtures incl.)')


In [ ]:
# Show the contents of every newly-written or modified file. Truncates
# large outputs to 4000 chars each.
MAX_CHARS = 4000
for f in diff['new'] + diff['modified']:
    full = workspace / f
    print('=' * 60)
    print(f'FILE: {f}')
    print('=' * 60)
    try:
        text = full.read_text(encoding='utf-8', errors='replace')
    except Exception as exc:
        print(f'(could not read: {exc})')
        continue
    if len(text) > MAX_CHARS:
        print(text[:MAX_CHARS])
        print(f'... [+{len(text)-MAX_CHARS} chars truncated]')
    else:
        print(text)
    print()


In [ ]:
print('--- AGENT FINAL ANSWER ---')
print(rec.final_answer or '(empty)')


## 9. Grading

Identisch zur PinchBench-Scoring-Logik (``evals/pinchbench/grading.py``):

- **automated**: ``def grade(transcript, workspace_path)`` aus Markdown
  in Subprocess ausgefuehrt, JSON-Score pro Kriterium.
- **hybrid**: automated UND llm_judge, beide Komponenten gemittelt.
- **llm_judge**: nur die LLM-Judge-Bewertung.

``USE_LLM_JUDGE`` toggelt den (teureren) LLM-Judge.


In [ ]:
from evals.pinchbench.transcript import build_transcript

transcript = build_transcript(rec.events, target.prompt)
print(f'Transcript entries: {len(transcript)}')

# Quick visual: count content-type per role.
from collections import Counter
breakdown = Counter()
for e in transcript:
    msg = e.get('message') or {}
    role = msg.get('role', '?')
    for c in (msg.get('content') or []):
        breakdown[(role, c.get('type'))] += 1
for (role, ctype), n in sorted(breakdown.items(), key=lambda x: -x[1]):
    print(f'  {role:10s}  {ctype:14s}  {n:4d}')


In [ ]:
USE_LLM_JUDGE = False   # set True to run the LLM-judge

from evals.pinchbench.grading import run_automated_check, aggregate_scores, run_llm_judge

components = {}
details = {'task_id': target.id, 'grading_type': target.grading_type}

# Automated
if target.grading_type in ('automated', 'hybrid') and target.grade_function:
    print(f'Running automated grader (timeout={min(30, target.timeout_seconds)}s)...')
    auto = run_automated_check(
        target.grade_function, transcript, workspace,
        timeout_seconds=min(30, target.timeout_seconds),
    )
    if auto.get('ok'):
        scores = auto.get('scores') or {}
        components['automated'] = aggregate_scores(scores)
        details['automated_scores'] = scores
        print(f'  ok, mean={components["automated"]:.3f}')
        for k, v in scores.items():
            marker = '[ok]' if v >= 0.99 else ('[..]' if v >= 0.5 else '[!!]')
            print(f'    {marker} {k:30s}  {v:.2f}')
    else:
        details['automated_error'] = auto.get('error', 'unknown')
        print(f'  FAILED: {auto.get("error")[:300]}')
        if 'traceback' in auto:
            print(auto['traceback'][:1500])
else:
    print('(no automated grader for this task)')

# LLM judge
if USE_LLM_JUDGE and target.grading_type in ('llm_judge', 'hybrid'):
    print()
    print('Running LLM judge...')
    judge = await run_llm_judge(
        prompt=target.prompt,
        transcript=transcript,
        rubric=target.llm_judge_rubric,
        expected_behavior=target.expected_behavior,
        grading_criteria=target.grading_criteria,
    )
    if judge.get('ok'):
        components['judge'] = float(judge['score'])
        details['judge_reasoning'] = judge.get('reasoning', '')
        print(f'  ok, score={components["judge"]:.3f}')
        print(f'  reasoning: {judge.get("reasoning", "")}')
    else:
        details['judge_error'] = judge.get('error', '')
        print(f'  FAILED: {judge.get("error")[:300]}')

print()
print('=' * 60)
if components:
    final = sum(components.values()) / len(components)
    print(f'FINAL SCORE: {final*100:.1f}%   components={components}')
else:
    print('FINAL SCORE: 0.0%   (no usable grader)')


## 10. Failure-Analyse

Wenn der Score < 1.0: was genau fehlt?


In [ ]:
# Re-run only what was missing - cheap diagnostic.
auto_scores = details.get('automated_scores') or {}
if auto_scores:
    missing = [(k, v) for k, v in auto_scores.items() if v < 1.0]
    if missing:
        print('FAILED / PARTIAL criteria:')
        for k, v in missing:
            print(f'  {k:35s} = {v:.2f}')
    else:
        print('All automated criteria passed.')

# Pivot/deliverable diagnostics
writes_via_tool = rec.tool_calls.get('file_write', 0) + rec.tool_calls.get('edit', 0)
sub_agent_calls = rec.tool_calls.get('call_agents_parallel', 0)
print()
print(f'tool_calls          : {dict(rec.tool_calls)}')
print(f'file_write+edit     : {writes_via_tool}')
print(f'call_agents_parallel: {sub_agent_calls}')
print(f'event_count         : {len(rec.events)}')
print(f'duration            : {rec.duration:.1f}s')
print(f'llm_calls           : {rec.llm_calls}')
print(f'stream_restarts     : {len(rec.stream_restarts)}')
if rec.stream_restarts:
    for sr in rec.stream_restarts:
        print(f'  - stage={sr["stage"]} reason={sr["reason"]}')

# Files actually written
if not diff['new'] and not diff['modified']:
    print()
    print('No new or modified files in workspace - deliverable was never written.')
elif diff['new']:
    print()
    print('Files agent wrote:')
    for f in diff['new']:
        full = workspace / f
        size = full.stat().st_size if full.exists() else 0
        print(f'  {f}  ({size} bytes)')


## 11. Cleanup

Schliesst den Agent, raeumt das per-Task-Workspace auf. Wiederhole
diese Zelle wenn du den naechsten Task untersuchen willst (oder
einfach Section 4-10 nochmal mit anderer ``TARGET_TASK_ID``).


In [ ]:
import shutil
await agent.close()
if workspace.name.startswith('pinchbench_ws_') and workspace.exists():
    shutil.rmtree(workspace, ignore_errors=True)
    print(f'Cleaned up workspace: {workspace}')
print('Done. Pick a new TARGET_TASK_ID and rerun Section 4+.')


## 12. Optional: Kategorie-Mini-Sweep

Wenn du ein Pattern verstehst, lass alle Tasks einer Kategorie nacheinander
laufen und sieh, ob das Verhalten konsistent ist. Achtung: das ist viel
LLM-Time. Ein einzelner Lauf gibt dir die Daten in 30s-3min; eine
Kategorie mit n=28 dauert eher 30-90 min.


In [ ]:
CATEGORY_TO_SWEEP = None   # set to e.g. 'log_analysis' to actually run; None = noop

if CATEGORY_TO_SWEEP:
    sweep_tasks = [t for t in tasks if t.category == CATEGORY_TO_SWEEP]
    print(f'Sweep {CATEGORY_TO_SWEEP}: {len(sweep_tasks)} tasks')
    sweep_results = []
    for st in sweep_tasks:
        sweep_ws = _provision_workspace(list(st.workspace_files))
        sweep_prompt = _augment_prompt(st.prompt, sweep_ws)
        sweep_agent = await build_agent()
        try:
            sweep_rec = await alib.run(
                executor, sweep_agent, sweep_prompt,
                project_root=sweep_ws, snapshot_subdirs=('',),
                max_print_events=0, silent=True,
            )
            sweep_transcript = build_transcript(sweep_rec.events, st.prompt)
            sweep_score = None
            if st.grade_function:
                a = run_automated_check(st.grade_function, sweep_transcript, sweep_ws,
                                        timeout_seconds=min(30, st.timeout_seconds))
                if a.get('ok'):
                    sweep_score = aggregate_scores(a.get('scores') or {})
            sweep_results.append({
                'id': st.id, 'score': sweep_score,
                'events': len(sweep_rec.events), 'tools': sum(sweep_rec.tool_calls.values()),
                'writes': sweep_rec.tool_calls.get('file_write', 0) + sweep_rec.tool_calls.get('edit', 0),
                'duration': sweep_rec.duration,
            })
            await sweep_agent.close()
        finally:
            import shutil as _sh
            if sweep_ws.name.startswith('pinchbench_ws_') and sweep_ws.exists():
                _sh.rmtree(sweep_ws, ignore_errors=True)
    import pandas as pd
    sweep_df = pd.DataFrame(sweep_results).sort_values('score', na_position='last')
    print(sweep_df.to_string(index=False))
else:
    print('(noop - set CATEGORY_TO_SWEEP to run)')
